### 数据集下载

https://archive.ics.uci.edu/dataset/15/breast+cancer+wisconsin+original

In [1]:
import pandas as pd

# 先不指定列名（原数据无表头），以便查看基本信息
df = pd.read_csv(
    'data/breast-cancer-wisconsin.csv',
    header=None
)

# 查看数据的前几行
print(df.head())

# 查看数据基本信息
print(df.info())

# 查看数据集的形状
print("数据集形状：", df.shape)


        0   1   2   3   4   5   6   7   8   9   10
0  1000025   5   1   1   1   2   1   3   1   1   2
1  1002945   5   4   4   5   7  10   3   2   1   2
2  1015425   3   1   1   1   2   2   3   1   1   2
3  1016277   6   8   8   1   3   4   3   7   1   2
4  1017023   4   1   1   3   2   1   3   1   1   2
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699 entries, 0 to 698
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       699 non-null    int64 
 1   1       699 non-null    int64 
 2   2       699 non-null    int64 
 3   3       699 non-null    int64 
 4   4       699 non-null    int64 
 5   5       699 non-null    int64 
 6   6       699 non-null    object
 7   7       699 non-null    int64 
 8   8       699 non-null    int64 
 9   9       699 non-null    int64 
 10  10      699 non-null    int64 
dtypes: int64(10), object(1)
memory usage: 60.2+ KB
None
数据集形状： (699, 11)


In [2]:
# 发现列6数据类型非整型，可能存在缺失值

# 输出列6含有的所有值
print(df[6].unique())

['1' '10' '2' '4' '3' '9' '7' '?' '5' '8' '6']


In [3]:
# 将缺失值（用'?'表示）替换为NaN
df.replace('?', pd.NA, inplace=True)

# 查看每列缺失值数量
print("每列缺失值数量：")
print(df.isna().sum())

# 直接删除含有缺失值的行
df.dropna(inplace=True)

# 再次检查是否存在缺失值
print("再次检查每列缺失值数量：")
print(df.isna().sum())

# 查看数据集形状
print("处理缺失值后数据集形状：", df.shape)



每列缺失值数量：
0      0
1      0
2      0
3      0
4      0
5      0
6     16
7      0
8      0
9      0
10     0
dtype: int64
再次检查每列缺失值数量：
0     0
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     0
9     0
10    0
dtype: int64
处理缺失值后数据集形状： (683, 11)


In [7]:
# 定义列名，因为原始数据集没有表头
column_names = [
    'id', 'clump_thickness', 'uniformity_of_cell_size',
    'uniformity_of_cell_shape', 'marginal_adhesion',
    'single_epithelial_cell_size', 'bare_nuclei',
    'bland_chromatin', 'normal_nucleoli', 'mitoses', 'class'
]
df.columns = column_names

# 查看数据集的前几行
print(df.head())


        id  clump_thickness  uniformity_of_cell_size  \
0  1000025                5                        1   
1  1002945                5                        4   
2  1015425                3                        1   
3  1016277                6                        8   
4  1017023                4                        1   

   uniformity_of_cell_shape  marginal_adhesion  single_epithelial_cell_size  \
0                         1                  1                            2   
1                         4                  5                            7   
2                         1                  1                            2   
3                         8                  1                            3   
4                         1                  3                            2   

  bare_nuclei  bland_chromatin  normal_nucleoli  mitoses  class  
0           1                3                1        1      2  
1          10                3                2        1

In [8]:
# 确定特征值及标签
X = df.iloc[:, 1:-1]  # 取除id和class以外的所有特征列
y = df['class']       # 标签为class（2-良性，4-恶性）

# 将标签转换为0和1（便于后续建模，2->0（良性），4->1（恶性））
y = y.replace({2: 0, 4: 1})

# 划分训练集与测试集
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
    # stratify=y 参数的作用是按照 y 标签的类别分布进行分层抽样划分训练集和测试集。
    # 这样可以保证训练集和测试集中的各类别比例与原始数据集保持一致，防止因数据不均衡导致模型训练或评估产生偏差。

# 进行标准化
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 创建逻辑回归模型
logreg = LogisticRegression(random_state=42, max_iter=1000)

# 训练模型
logreg.fit(X_train_scaled, y_train)

# 在测试集上进行预测
y_pred = logreg.predict(X_test_scaled)

# 输出预测结果
print("预测值：", y_pred)
print("真实值：", y_test.values)

# 评估模型
acc = accuracy_score(y_test, y_pred)
print("准确率：", acc)

# 输出混淆矩阵
cm = confusion_matrix(y_test, y_pred)
print("混淆矩阵：\n", cm)

# 输出详细分类报告
print("分类报告：\n", classification_report(y_test, y_pred))


预测值： [0 0 0 0 0 1 0 0 1 0 1 0 0 1 1 0 0 1 0 1 0 1 1 0 1 0 1 0 0 0 0 1 0 1 0 0 1
 0 0 1 1 1 0 0 0 0 1 1 1 0 1 0 0 1 0 0 1 0 0 0 0 1 0 0 1 1 0 0 0 0 0 0 1 0
 0 0 1 0 0 1 0 0 0 0 1 0 0 0 1 0 1 0 0 1 0 0 0 0 1 1 0 0 1 0 0 0 1 1 0 1 1
 0 0 1 1 0 1 0 1 1 1 1 0 0 1 0 1 0 0 0 0 0 1 0 0 0 1]
真实值： [0 0 0 0 0 1 0 0 1 0 1 0 0 1 1 0 0 1 0 1 0 1 1 0 1 0 1 0 0 0 0 1 0 1 0 0 1
 0 0 1 1 0 0 0 0 0 0 1 1 0 1 0 0 1 0 0 1 0 0 0 0 1 0 0 1 1 0 0 0 0 0 0 1 0
 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 1 0 0 1 0 0 0 0 1 1 0 0 1 0 0 0 0 1 0 1 1
 0 0 1 1 1 1 0 1 1 1 1 0 0 1 0 1 0 0 0 0 0 1 0 0 0 1]
准确率： 0.9635036496350365
混淆矩阵：
 [[85  4]
 [ 1 47]]
分类报告：
               precision    recall  f1-score   support

           0       0.99      0.96      0.97        89
           1       0.92      0.98      0.95        48

    accuracy                           0.96       137
   macro avg       0.95      0.97      0.96       137
weighted avg       0.96      0.96      0.96       137

